In [3]:
# File: project-folder/model/train_models.py
import sys
import subprocess

# Automatically check and install missing dependencies from requirements.txt
def install_requirements():
    try:
        import pandas
        import sklearn
        import joblib
    except ImportError:
        print("Installing missing dependencies from requirements.txt...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])

install_requirements()

import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

def create_synthetic_bank_data(n_samples=20000, random_state=42):
    """Generates synthetic dataset following UCI Bank Marketing Schema if raw dataset is not found."""
    np.random.seed(random_state)
    jobs = ['admin.', 'blue-collar', 'entrepreneur', 'housemaid', 'management', 'retired', 'self-employed', 'services', 'student', 'technician', 'unemployed', 'unknown']
    maritals = ['divorced', 'married', 'single', 'unknown']
    educations = ['basic.4y', 'basic.6y', 'basic.9y', 'high.school', 'illiterate', 'professional.course', 'university.degree', 'unknown']
    defaults = ['no', 'yes', 'unknown']
    housings = ['no', 'yes', 'unknown']
    loans = ['no', 'yes', 'unknown']
    contacts = ['cellular', 'telephone']
    months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
    poutcomes = ['failure', 'nonexistent', 'success']

    data = {
        'age': np.random.randint(18, 90, size=n_samples),
        'job': np.random.choice(jobs, size=n_samples),
        'marital': np.random.choice(maritals, size=n_samples),
        'education': np.random.choice(educations, size=n_samples),
        'default': np.random.choice(defaults, p=[0.8, 0.05, 0.15], size=n_samples),
        'housing': np.random.choice(housings, p=[0.45, 0.5, 0.05], size=n_samples),
        'loan': np.random.choice(loans, p=[0.8, 0.15, 0.05], size=n_samples),
        'contact': np.random.choice(contacts, size=n_samples),
        'month': np.random.choice(months, size=n_samples),
        'day': np.random.randint(1, 31, size=n_samples),
        'duration': np.random.exponential(scale=250, size=n_samples).astype(int),
        'campaign': np.random.randint(1, 10, size=n_samples),
        'pdays': np.random.choice([-1, 999, 3, 6, 12], p=[0.7, 0.2, 0.03, 0.04, 0.03], size=n_samples),
        'previous': np.random.poisson(0.5, size=n_samples),
        'poutcome': np.random.choice(poutcomes, p=[0.1, 0.8, 0.1], size=n_samples)
    }

    df = pd.DataFrame(data)
    logit = (-3.0 
             + 0.005 * df['duration'] 
             + 1.5 * (df['poutcome'] == 'success') 
             + 0.02 * df['age'] 
             - 0.1 * df['campaign'])
    prob = 1 / (1 + np.exp(-logit))
    df['y'] = np.where(np.random.rand(n_samples) < prob, 'yes', 'no')
    return df

def train_and_save_pipeline(data_path="bank-full.csv", output_dir=None):
    # Determine execution path safely
    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()

    if output_dir is None:
        output_dir = base_dir

    os.makedirs(output_dir, exist_ok=True)
    
    parent_dir = os.path.dirname(base_dir)
    dataset_file = os.path.join(parent_dir, data_path)

    if os.path.exists(dataset_file):
        print(f"Loading data from {dataset_file}...")
        df = pd.read_csv(dataset_file, sep=";")
    elif os.path.exists(data_path):
        print(f"Loading data from {data_path}...")
        df = pd.read_csv(data_path, sep=";")
    else:
        print("Local bank-full.csv not found. Generating dataset of 20,000 samples...")
        df = create_synthetic_bank_data(n_samples=20000)

    # Constraint: Limit to 20,000 entries
    df = df.iloc[:20000].copy()
    
    # Save a slice for test_data.csv in the root folder
    test_sample = df.sample(n=1000, random_state=42)
    test_csv_path = os.path.join(parent_dir, "test_data.csv")
    test_sample.to_csv(test_csv_path, index=False)
    print(f"Saved test_data.csv to {test_csv_path}")

    X = df.drop(columns=['y'])
    y = (df['y'] == 'yes').astype(int)

    num_cols = ['age', 'day', 'duration', 'campaign', 'pdays', 'previous']
    cat_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
        ]
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    X_train_trans = preprocessor.fit_transform(X_train)
    X_test_trans = preprocessor.transform(X_test)

    joblib.dump(preprocessor, os.path.join(output_dir, "preprocessor.pkl"))

    models = {
        'logistic_regression': LogisticRegression(max_iter=1000, random_state=42),
        'decision_tree': DecisionTreeClassifier(max_depth=8, random_state=42),
        'knn': KNeighborsClassifier(n_neighbors=7),
        'naive_bayes': GaussianNB(),
        'random_forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    }

    metrics_records = []

    for name, model in models.items():
        model.fit(X_train_trans, y_train)
        joblib.dump(model, os.path.join(output_dir, f"{name}.pkl"))

        y_pred = model.predict(X_test_trans)
        y_prob = model.predict_proba(X_test_trans)[:, 1] if hasattr(model, "predict_proba") else y_pred

        acc = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        mcc = matthews_corrcoef(y_test, y_pred)

        metrics_records.append({
            'ML Model Name': name.replace('_', ' ').title(),
            'Accuracy': round(acc, 4),
            'AUC': round(auc, 4),
            'Precision': round(prec, 4),
            'Recall': round(rec, 4),
            'F1': round(f1, 4),
            'MCC': round(mcc, 4)
        })

    summary_df = pd.DataFrame(metrics_records)
    print("\n--- Model Training Completed Successfully ---")
    print(summary_df.to_string(index=False))

if __name__ == "__main__":
    train_and_save_pipeline()

Loading data from D:\projects\ml\Assignment2\min\bank-full.csv...
Saved test_data.csv to D:\projects\ml\Assignment2\min\test_data.csv

--- Model Training Completed Successfully ---
      ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
Logistic Regression    0.9560 0.9614     0.5464  0.2865 0.3759 0.3754
      Decision Tree    0.9537 0.8868     0.5000  0.4108 0.4510 0.4294
                Knn    0.9570 0.8642     0.5823  0.2486 0.3485 0.3623
        Naive Bayes    0.9045 0.8833     0.2592  0.5730 0.3569 0.3421
      Random Forest    0.9573 0.9558     0.6750  0.1459 0.2400 0.3009
